In [22]:
# =============================================================================
#  UTILITAS BERSAMA
#  Dipakai oleh tahap training, inferensi, dan export hasil
# =============================================================================

import re
import warnings
import pandas as pd
import joblib
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")

MODEL_PATH = "model_sqli_nb.pkl"

STOPWORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for",
    "of", "and", "with", "by", "as", "be", "was", "are", "were",
    "this", "that", "have", "has", "had", "do", "does", "did",
    "but", "so", "if", "then", "than", "its", "into", "from",
    "there", "their", "they", "will", "would", "could", "should"
}

SQL_KEYWORDS = {
    "select", "from", "where", "and", "or", "not", "is", "in",
    "like", "union", "insert", "update", "delete", "drop", "create",
    "table", "into", "values", "order", "by", "group", "having",
    "join", "on", "null", "true", "false", "case", "when", "then",
    "else", "end", "limit", "offset", "between", "exists", "all",
    "distinct", "count", "sum", "max", "min", "avg", "sleep",
    "benchmark", "char", "concat", "substring", "load_file",
    "outfile", "exec", "execute", "cast", "convert", "if"
}

def preprocess_text(text):
    """
    Fungsi preprocessing teks untuk deteksi SQL Injection.

    Tahapan:
    1. Lowercase
    2. Tokenisasi dengan regex (pisahkan kata & simbol SQL penting)
    3. Hapus stopword non-SQL
    4. Gabungkan kembali menjadi string bersih
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    text = re.sub(r'\d+', '0', text)

    tokens = re.findall(
        r"[a-z0-9_]+|--|/\*|\*/|'|\"|\(|\)|=|<|>|;|#|,|\*|\+|-|%",
        text
    )

    filtered = [
        token for token in tokens
        if token not in STOPWORDS or token in SQL_KEYWORDS
    ]

    return " ".join(filtered)


In [23]:
# =============================================================================
#  UTILITAS BERSAMA
#  Dipakai oleh tahap training, inferensi, dan export hasil
# =============================================================================

import re
import warnings
import numpy as np
import pandas as pd
import joblib
import time
from IPython.display import display
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from rapidfuzz import fuzz

warnings.filterwarnings("ignore")

MODEL_PATH = "model_sqli_nb.pkl"

STOPWORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for",
    "of", "and", "with", "by", "as", "be", "was", "are", "were",
    "this", "that", "have", "has", "had", "do", "does", "did",
    "but", "so", "if", "then", "than", "its", "into", "from",
    "there", "their", "they", "will", "would", "could", "should"
}

SQL_KEYWORDS = {
    "select", "from", "where", "and", "or", "not", "is", "in",
    "like", "union", "insert", "update", "delete", "drop", "create",
    "table", "into", "values", "order", "by", "group", "having",
    "join", "on", "null", "true", "false", "case", "when", "then",
    "else", "end", "limit", "offset", "between", "exists", "all",
    "distinct", "count", "sum", "max", "min", "avg", "sleep",
    "benchmark", "char", "concat", "substring", "load_file",
    "outfile", "exec", "execute", "cast", "convert", "if"
}

def preprocess_text(text):
    """
    Fungsi preprocessing teks untuk deteksi SQL Injection dengan Generalisasi Struktural.

    Tahapan:
    1. Lowercase
    2. Generalisasi String Literal (mengubah teks di dalam kutip menjadi 'str')
    3. Generalisasi Angka (mengubah angka menjadi '0')
    4. Tokenisasi dengan regex (pisahkan kata & simbol SQL penting)
    5. Hapus stopword non-SQL
    6. Gabungkan kembali menjadi string bersih
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Generalisasi nilai di dalam kutip tunggal (') dan ganda (")
    text = re.sub(r"'(.*?)'", "'str'", text)
    text = re.sub(r'"(.*?)"', '"str"', text)
    
    # 3. Generalisasi seluruh numerik/angka menjadi '0'
    text = re.sub(r'\d+', '0', text)

    # 4. Tokenisasi regex khusus SQL (Perbaikan backslash)
    tokens = re.findall(
        r"[a-z0-9_]+|--|/\*|\*/|'|\"|\(|\)|=|<|>|;|#|,|\*|\+|-|%",
        text
    )

    # 5. Filter stopword dengan mempertahankan keyword SQL penting
    filtered = [
        token for token in tokens
        if token not in STOPWORDS or token in SQL_KEYWORDS
    ]

    return " ".join(filtered)

In [24]:
# =============================================================================
#  TAHAP 1 — MEMUAT DATASET
# =============================================================================

print("=" * 65)
print("  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES")
print("=" * 65)

# DATASET_PATH = "drive/MyDrive/rbsqli_dataset.csv"
DATASET_PATH = "rbsqli_dataset.csv"

df = pd.read_csv(DATASET_PATH)

# Membaca berdasarkan indeks kolom (0 = input Query, 1 = tipe serangan, 2 = Label)
df = df.iloc[:, [0, 1, 2]]

# Menggunakan konvensi nama akademis dan memperbaiki typo koma
df.columns = ["Query", "Category", "Label"]

print(f"\n[1] DATASET DIMUAT")
print(f"    Total data mentah : {len(df)} baris")
print(f"    Kolom             : {list(df.columns)}")

  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES

[1] DATASET DIMUAT
    Total data mentah : 10190450 baris
    Kolom             : ['Query', 'Category', 'Label']


In [25]:
print("\n[Informasi DataFrame df]")
print(f"Jumlah Baris    : {df.shape[0]} baris")
print(f"Jumlah Kolom    : {df.shape[1]} kolom")
print("\nRingkasan Informasi (df.info()):")

# JIKA Anda tetap menggunakan nama kolom 'category', gunakan baris ini:
kategori_unik = df['Category'].unique().tolist()

print("Daftar seluruh kategori yang ada:")
print(kategori_unik)
df.info()



[Informasi DataFrame df]
Jumlah Baris    : 10190450 baris
Jumlah Kolom    : 3 kolom

Ringkasan Informasi (df.info()):
Daftar seluruh kategori yang ada:
['None_Type', 'Error-Based', 'meta_based', 'stackqueries_based', 'Time-Based', 'Union-Based', 'boolean-based']
<class 'pandas.DataFrame'>
RangeIndex: 10190450 entries, 0 to 10190449
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   Query     str  
 1   Category  str  
 2   Label     str  
dtypes: str(3)
memory usage: 233.2 MB


In [26]:
# =============================================================================
#  TAHAP 2 — PREPROCESSING OTOMATIS (DATA CLEANSING)
# =============================================================================

print("\n[2] PREPROCESSING OTOMATIS")

sebelum = len(df)

# Hapus baris dengan nilai kosong pada kolom utama
df.dropna(subset=["Query", "Label"], inplace=True)
print(f"    Hapus baris kosong       : {sebelum - len(df)} baris dihapus")

# Normalisasi Label hanya sekali
df["Label"] = (
    df["Label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"yes": "1", "no": "0"})
)

# Pastikan hanya bernilai 0 atau 1
df = df[df["Label"].str.match(r"^[01]$")]
df["Label"] = df["Label"].astype(int)
print(f"    Setelah filter Label     : {len(df)} baris valid")

# Terapkan preprocessing ke kolom Query
df["Query_clean"] = df["Query"].apply(preprocess_text)

# Hapus duplikasi berdasarkan teks yang sudah dibersihkan
sebelum_dup = len(df)
df.drop_duplicates(subset=["Query_clean"], inplace=True)
df = df[df["Query_clean"].str.strip() != ""]
print(f"    Duplikasi dihapus        : {sebelum_dup - len(df)} baris")

# Reset index setelah pembersihan
df.reset_index(drop=True, inplace=True)

print(f"    Total data bersih        : {len(df)} baris")
print(f"\n    Distribusi Kelas:")
distribusi = df["Label"].value_counts()
print(f"      Label 0 (Normal) : {distribusi.get(0, 0)} sampel")
print(f"      Label 1 (SQLI)   : {distribusi.get(1, 0)} sampel")


[2] PREPROCESSING OTOMATIS
    Hapus baris kosong       : 0 baris dihapus
    Setelah filter Label     : 10190450 baris valid
    Duplikasi dihapus        : 8381829 baris
    Total data bersih        : 1808621 baris

    Distribusi Kelas:
      Label 0 (Normal) : 959040 sampel
      Label 1 (SQLI)   : 849581 sampel


In [27]:
# =============================================================================
#  TAHAP 3 — RINGKASAN HASIL PREPROCESSING
# =============================================================================

print("\n[3] TEXT PREPROCESSING SELESAI")
print("    Contoh hasil preprocessing:")
for i in range(min(3, len(df))):
    print(f"\n    [{i+1}] Asli   : {df['Query'].iloc[i][:70]}")
    print(f"         Bersih : {df['Query_clean'].iloc[i][:70]}")
    print(f"         Label  : {'SQLI (1)' if df['Label'].iloc[i] == 1 else 'Normal (0)'}")


[3] TEXT PREPROCESSING SELESAI
    Contoh hasil preprocessing:

    [1] Asli   : UPDATE id, email FROM login WHERE created_at LIKE '2024-01-01' OR crea
         Bersih : update id , email from login where created_at like ' str ' or created_
         Label  : Normal (0)

    [2] Asli   : UPDATE MIN(created_at) FROM payments WHERE product_id BETWEEN 1 AND pr
         Bersih : update min ( created_at ) from payments where product_id between 0 and
         Label  : Normal (0)

    [3] Asli   : EXEC created_at, updated_at FROM admin WHERE price LIKE 1 NOT price LI
         Bersih : exec created_at , updated_at from admin where price like 0 not price l
         Label  : Normal (0)


In [28]:
# =============================================================================
#  TAHAP tambahan — BALANCING KELAS & DOWNSAMPLING PER KATEGORI (DINAMIS)
# =============================================================================

# Tentukan target ideal jumlah baris per kategori target
TARGET_PER_KATEGORI = 15000
KATEGORI_TARGET = ['Error-Based', 'meta_based', 'stackqueries_based', 'Time-Based', 'Union-Based', 'boolean-based']

print("\n[tambahan] BALANCING & DOWNSAMPLING KELAS")

# Hitung ketersediaan baris unik asli pasca-generalisasi
jumlah_per_kat = {}
for kat in KATEGORI_TARGET:
    jumlah_per_kat[kat] = (df["Category"] == kat).sum()
    print(f"    Tersedia setelah deduplikasi struktural '{kat}': {jumlah_per_kat[kat]} baris")

# Optimasi Akademis: Menyesuaikan target sampling secara otomatis berdasarkan data terkecil agar tidak crash
min_tersedia = min(jumlah_per_kat.values())
TARGET_RIIL = min(TARGET_PER_KATEGORI, min_tersedia)
print(f"\n[Info] Target disesuaikan menjadi {TARGET_RIIL} baris per kategori SQLi untuk menghindari sample error.")

sampled_dfs = []

# Sampling masing-masing kategori target SQLi
for kat in KATEGORI_TARGET:
    df_kat = df[df["Category"] == kat].sample(n=TARGET_RIIL, random_state=42)
    sampled_dfs.append(df_kat)

# Menyeimbangkan data Normal (Label 0) dengan total akumulasi dari seluruh kategori SQLi yang diambil
TARGET_NORMAL = TARGET_RIIL * len(KATEGORI_TARGET)
df_normal = df[df["Label"] == 0].sample(n=TARGET_NORMAL, random_state=42)
sampled_dfs.append(df_normal)

# Gabungkan semua subset data dan acak urutannya (shuffle)
df_balanced = pd.concat(sampled_dfs, axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
df = df_balanced.copy()

print(f"\n    Berhasil di-balancing berdasarkan struktur query.")
print(f"    Total data gabungan setelah balancing: {len(df)} baris")

print("\n    Distribusi kategori setelah balancing:")
print(df["Category"].value_counts())


[tambahan] BALANCING & DOWNSAMPLING KELAS
    Tersedia setelah deduplikasi struktural 'Error-Based': 71082 baris
    Tersedia setelah deduplikasi struktural 'meta_based': 334793 baris
    Tersedia setelah deduplikasi struktural 'stackqueries_based': 94207 baris
    Tersedia setelah deduplikasi struktural 'Time-Based': 291383 baris
    Tersedia setelah deduplikasi struktural 'Union-Based': 13572 baris
    Tersedia setelah deduplikasi struktural 'boolean-based': 44544 baris

[Info] Target disesuaikan menjadi 13572 baris per kategori SQLi untuk menghindari sample error.

    Berhasil di-balancing berdasarkan struktur query.
    Total data gabungan setelah balancing: 162864 baris

    Distribusi kategori setelah balancing:
Category
None_Type             81432
Error-Based           13572
Time-Based            13572
meta_based            13572
Union-Based           13572
boolean-based         13572
stackqueries_based    13572
Name: count, dtype: int64


In [29]:
# =============================================================================
#  TAHAP 4 — SPLIT DATA DAN VERIFIKASI DATA LEAKAGE
# =============================================================================
from sklearn.model_selection import train_test_split

source_df = df_balanced if "df_balanced" in globals() else df
print(f"\n[4] SPLIT DATASET (80:20)")
print(f"    Total data yang dipakai: {len(source_df)} sampel")

X = source_df["Query_clean"].astype(str)
y = source_df["Label"]

# Pembagian data training dan testing secara stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"    Data Training : {len(X_train)} sampel")
print(f"    Data Testing  : {len(X_test)} sampel")

print("\n    Cek Exact Overlap (Data Leakage):")
train_set = set(X_train)
test_set = set(X_test)
overlap = len(train_set.intersection(test_set))
print(f"    Irisan train ∩ test      : {overlap} (harus 0 — {'✅ Aman' if overlap == 0 else '⚠️ Ada Leakage!'})")

print("\n    Cek Near-Duplicate (threshold similarity > 0.95):")
sample_test = X_test.sample(n=min(300, len(X_test)), random_state=42).tolist()
sample_train = X_train.sample(n=min(500, len(X_train)), random_state=42).tolist()

count_similar = 0
for t in sample_test:
    for tr in sample_train:
        if fuzz.ratio(t, tr) > 95:
            count_similar += 1
            break

print(f"    Near-duplicate ditemukan : {count_similar} dari {len(sample_test)} sampel test")


[4] SPLIT DATASET (80:20)
    Total data yang dipakai: 162864 sampel
    Data Training : 130291 sampel
    Data Testing  : 32573 sampel

    Cek Exact Overlap (Data Leakage):
    Irisan train ∩ test      : 0 (harus 0 — ✅ Aman)

    Cek Near-Duplicate (threshold similarity > 0.95):
    Near-duplicate ditemukan : 8 dari 300 sampel test


In [30]:
# =============================================================================
#  TAHAP 5 — DEFINISI PIPELINE TF-IDF + MULTINOMIAL NAÏVE BAYES
# =============================================================================

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=5,
            max_features=10000,
            sublinear_tf=True
        )
    ),
    (
        "nb",
        MultinomialNB(alpha=0.1)
    )
])

print("\n[5] PIPELINE DIDEFINISIKAN")
print("    Algoritma  : Multinomial Naïve Bayes")
print("    Ekstraksi  : TF-IDF (Char n-gram 3-5, max 10000 fitur)")
print("    Alpha (smoothing) : 0.1")



[5] PIPELINE DIDEFINISIKAN
    Algoritma  : Multinomial Naïve Bayes
    Ekstraksi  : TF-IDF (Char n-gram 3-5, max 10000 fitur)
    Alpha (smoothing) : 0.1


In [31]:
# =============================================================================
#  TAHAP 6 — CROSS-VALIDATION STRATIFIED
# =============================================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print(f"\n[6] CROSS-VALIDATION (5-Fold Stratified)")
print(f"    F1 per Fold : {[round(s, 4) for s in scores]}")
print(f"    Rata-rata   : {scores.mean():.4f}")
print(f"    Std Dev     : {scores.std():.4f}")



[6] CROSS-VALIDATION (5-Fold Stratified)
    F1 per Fold : [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
    Rata-rata   : 1.0000
    Std Dev     : 0.0000


In [32]:
# =============================================================================
#  TAHAP 7 — TRAINING MODEL
# =============================================================================

pipeline.fit(X_train, y_train)

print(f"\n[7] MODEL BERHASIL DILATIH")
print(f"    Jumlah data training : {len(X_train)} sampel")



[7] MODEL BERHASIL DILATIH
    Jumlah data training : 130291 sampel


In [33]:
# =============================================================================
#  TAHAP 8 — EVALUASI MODEL
# =============================================================================

y_pred = pipeline.predict(X_test)

akurasi = accuracy_score(y_test, y_pred)
presisi = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\n[8] HASIL EVALUASI MODEL")
print("    " + "─" * 40)
print(f"    Akurasi   : {akurasi * 100:.2f}%")
print(f"    Presisi   : {presisi * 100:.2f}%")
print(f"    Recall    : {recall * 100:.2f}%")
print(f"    F1-Score  : {f1 * 100:.2f}%")
print("    " + "─" * 40)

tn, fp, fn, tp = cm.ravel()
print(f"\n    CONFUSION MATRIX")
print(f"    {'':20} Prediksi Normal  Prediksi SQLI")
print(f"    {'Aktual Normal':<20} {tn:<17} {fp}")
print(f"    {'Aktual SQLI':<20} {fn:<17} {tp}")
print(f"\n      TP (Benar SQLI)    : {tp}")
print(f"      TN (Benar Normal)  : {tn}")
print(f"      FP (False Positive): {fp}")
print(f"      FN (False Negative): {fn}")

print("\n    CLASSIFICATION REPORT:")
print(classification_report(
    y_test, y_pred,
    target_names=["Normal (0)", "SQLI (1)"]
))



[8] HASIL EVALUASI MODEL
    ────────────────────────────────────────
    Akurasi   : 100.00%
    Presisi   : 100.00%
    Recall    : 100.00%
    F1-Score  : 100.00%
    ────────────────────────────────────────

    CONFUSION MATRIX
                         Prediksi Normal  Prediksi SQLI
    Aktual Normal        16287             0
    Aktual SQLI          0                 16286

      TP (Benar SQLI)    : 16286
      TN (Benar Normal)  : 16287
      FP (False Positive): 0
      FN (False Negative): 0

    CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     16287
    SQLI (1)       1.00      1.00      1.00     16286

    accuracy                           1.00     32573
   macro avg       1.00      1.00      1.00     32573
weighted avg       1.00      1.00      1.00     32573



In [34]:
# =============================================================================
#  TAHAP 9 — UJI MANUAL DETEKSI
# =============================================================================

def deteksi_sqli_dengan_waktu(input_teks: str) -> dict:
    """Fungsi deteksi SQL Injection dengan pengukuran waktu proses."""
    start_time = time.time()
    teks_bersih = preprocess_text(input_teks)
    prediksi = pipeline.predict([teks_bersih])[0]
    probabilitas = pipeline.predict_proba([teks_bersih])[0]
    latency_ms = (time.time() - start_time) * 1000

    return {
        "input": input_teks,
        "prediksi": "SQLI" if prediksi == 1 else "Normal",
        "prob_sqli": round(probabilitas[1] * 100, 2),
        "prob_normal": round(probabilitas[0] * 100, 2),
        "status": "🚫 DIBLOKIR" if prediksi == 1 else "✅ DIIZINKAN",
        "latency_ms": round(latency_ms, 2)
    }

sampel_uji = [
    "' OR '1'='1",
    "SELECT * FROM users WHERE id = 1",
    "UNION SELECT username, password FROM users--",
    "admin",
    "1; DROP TABLE users;--",
    "search=buku",
    "-1' UNION ALL SELECT NULL,NULL,NULL--",
    "username=malik&password=12345",
]

print("\n[9] UJI MANUAL DETEKSI")
print("    " + "─" * 60)

for sampel in sampel_uji:
    hasil = deteksi_sqli_dengan_waktu(sampel)
    print(f"\n    Input  : {hasil['input']}")
    print(f"    Status : {hasil['status']}")
    print(f"    P(SQLI)= {hasil['prob_sqli']}%  |  P(Normal)={hasil['prob_normal']}%")
    print(f"    Waktu  : {hasil['latency_ms']} ms  |  (Memenuhi target < 100ms)")

print("\n    " + "─" * 60)



[9] UJI MANUAL DETEKSI
    ────────────────────────────────────────────────────────────

    Input  : ' OR '1'='1
    Status : 🚫 DIBLOKIR
    P(SQLI)= 100.0%  |  P(Normal)=0.0%
    Waktu  : 2.8 ms  |  (Memenuhi target < 100ms)

    Input  : SELECT * FROM users WHERE id = 1
    Status : 🚫 DIBLOKIR
    P(SQLI)= 99.1%  |  P(Normal)=0.9%
    Waktu  : 5.12 ms  |  (Memenuhi target < 100ms)

    Input  : UNION SELECT username, password FROM users--
    Status : 🚫 DIBLOKIR
    P(SQLI)= 100.0%  |  P(Normal)=0.0%
    Waktu  : 4.39 ms  |  (Memenuhi target < 100ms)

    Input  : admin
    Status : ✅ DIIZINKAN
    P(SQLI)= 36.11%  |  P(Normal)=63.89%
    Waktu  : 2.45 ms  |  (Memenuhi target < 100ms)

    Input  : 1; DROP TABLE users;--
    Status : 🚫 DIBLOKIR
    P(SQLI)= 100.0%  |  P(Normal)=0.0%
    Waktu  : 1.8 ms  |  (Memenuhi target < 100ms)

    Input  : search=buku
    Status : 🚫 DIBLOKIR
    P(SQLI)= 62.68%  |  P(Normal)=37.32%
    Waktu  : 1.46 ms  |  (Memenuhi target < 100ms)

    Input

In [35]:
# =============================================================================
#  TAHAP 10 — SIMPAN MODEL
# =============================================================================

MODEL_PATH = "model_sqli_nb.pkl"
joblib.dump(pipeline, MODEL_PATH)

print(f"\n[8] MODEL DISIMPAN")
print(f"    Path  : {MODEL_PATH}")
print(f"    Muat kembali dengan: pipeline = joblib.load('{MODEL_PATH}')")
print("\n" + "=" * 65)
print("  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK")
print("=" * 65)


[8] MODEL DISIMPAN
    Path  : model_sqli_nb.pkl
    Muat kembali dengan: pipeline = joblib.load('model_sqli_nb.pkl')

  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK
